In [ ]:
from peft import LoraConfig

c:\Users\weich\anaconda3\envs\LLM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "../model/huggingface_transformer_format"

tokenizer = AutoTokenizer.from_pretrained(model_path)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float32
)

model.config.pad_token_id = tokenizer.pad_token_id

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:28<00:00,  5.18it/s, Materializing param=model.norm.weight]                              


In [4]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)
model.print_trainable_parameters(
)

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [ ]:
dataset = load_dataset("json", data_files=data_path)

In [ ]:
def format_prompt(example):
    return f"""### Prompt:
{example['prompt']}

### Response:
{example['response']}"""

def tokenize(example):
    text = format_prompt(example)

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize)

In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=0.0002,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="no"   # cleaner
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"]
)

trainer.train()

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
